# 00 — Pipeline Validation (Smoke Tests)

End-to-end pre-flight checks for every phase — verifies configs, connectivity, data,
and exercises `materialize_incremental` (single view) via CLI on the Feast pod to prove
the full Feast + Spark k8s:// + RAPIDS pipeline works.

**Key findings applied:**
- `feast apply` via `oc exec` on pod (gRPC remote registry strips backticks from SQL queries)
- `spark.sql.runSQLOnFiles: "true"` required in BOTH offline_store and batch_engine configs
- `get_historical_features` **now works** with Python-mode BFVs (patched SparkOfflineStore)

**Expected runtime: 3-5 minutes** (most time is executor pod startup)

In [ ]:
%pip install -q boto3 redis kubernetes feast pandas pyarrow s3fs fsspec kserve requests yamlmagic
%load_ext yamlmagic

In [ ]:
from _config import *
validate()

import time
_results = []

def check(phase, name, fn):
    """Run a check, capture pass/fail."""
    t0 = time.time()
    try:
        msg = fn()
        ms = (time.time() - t0) * 1000
        _results.append((phase, name, True, msg or "", ms))
        print(f"  ✓ {name} ({ms:.0f}ms){(' — ' + msg) if msg else ''}")
    except Exception as e:
        ms = (time.time() - t0) * 1000
        _results.append((phase, name, False, str(e), ms))
        print(f"  ✗ {name} ({ms:.0f}ms) — {e}")

---
## Phase 1: Data Pipeline

In [ ]:
import boto3, redis, sys, os
from kubernetes import client, config
from kubernetes.stream import stream

config.load_incluster_config()
v1 = client.CoreV1Api()

print("Phase 1: Data Pipeline\n")

# 1.1 S3 raw data
def check_s3_raw():
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                      aws_access_key_id=AWS_KEY, aws_secret_access_key=AWS_SECRET,
                      region_name="us-east-1")
    reviews = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/reviews/", MaxKeys=3)
    meta = s3.list_objects_v2(Bucket="smartshop-raw", Prefix="raw/metadata/", MaxKeys=3)
    n_rev = len(reviews.get("Contents", []))
    n_meta = len(meta.get("Contents", []))
    assert n_rev > 0, "No review files"
    assert n_meta > 0, "No metadata files"
    return f"{n_rev}+ review files, {n_meta}+ metadata files"

check(1, "S3 raw data exists", check_s3_raw)

# 1.2 Redis connectivity
def check_redis():
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT,
                    password=REDIS_PASSWORD or None, socket_timeout=5)
    r.ping()
    keys = r.dbsize()
    return f"{keys:,} keys"

check(1, "Redis connectivity", check_redis)

# 1.3 Feast pod running
def check_feast_pod():
    pods = v1.list_namespaced_pod(
        NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
        field_selector="status.phase=Running",
    ).items
    assert len(pods) > 0, "No running Feast pods"
    return pods[0].metadata.name

check(1, "Feast pod running", check_feast_pod)

# 1.4 Feature definitions importable
def check_features_import():
    sys.path.insert(0, os.path.join(os.getcwd(), "feature_repo"))
    from features import user_features, item_features, item_metadata
    assert user_features is not None
    return f"3 views: user_features, item_features, item_metadata"

check(1, "Feature definitions import", check_features_import)

# 1.5 Feast registry reachable (apply dry-run)
def check_feast_registry():
    from feast import FeatureStore
    store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)
    entities = store.list_entities()
    fvs = store.list_feature_views()
    return f"{len(entities)} entities, {len(fvs)} feature views registered"

check(1, "Feast registry reachable", check_feast_registry)

# 1.6 Spark on Feast pod can read S3
def check_spark_s3():
    pods = v1.list_namespaced_pod(
        NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
        field_selector="status.phase=Running",
    ).items
    pod_name = pods[0].metadata.name
    script = f"""
import sys
sys.path.insert(0, "{FEAST_FEATURE_REPO_ON_POD}")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[1]") \\
    .config("spark.driver.memory", "2g") \\
    .config("spark.driver.extraJavaOptions", "-Dcom.redhat.fips=false") \\
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio.{NAMESPACE}.svc.cluster.local:9000") \\
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \\
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \\
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider") \\
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \\
    .config("spark.sql.runSQLOnFiles", "true") \\
    .getOrCreate()
df = spark.sql("SELECT COUNT(*) AS n FROM parquet.`s3a://smartshop-raw/raw/reviews/Electronics/` LIMIT 1")
print(f"SPARK_OK:{{df.collect()[0]['n']}}")
spark.stop()
"""
    resp = stream(
        v1.connect_get_namespaced_pod_exec,
        pod_name, NAMESPACE, container="offline",
        command=["python3", "-u", "-c", script],
        stderr=True, stdout=True, stdin=False,
        _request_timeout=120,
    )
    assert "SPARK_OK:" in resp, f"Spark failed: {resp[-500:]}"
    count = resp.split("SPARK_OK:")[1].split()[0]
    return f"Electronics reviews: {int(count):,} rows readable"

check(1, "Spark → S3 on Feast pod", check_spark_s3)

# 1.7 feast-spark-driver Service exists (required for k8s:// executor→driver callbacks)
def check_spark_driver_svc():
    svc = v1.read_namespaced_service("feast-spark-driver", NAMESPACE)
    ports = {p.port for p in svc.spec.ports}
    assert 7078 in ports, "Port 7078 missing from feast-spark-driver service"
    assert 7079 in ports, "Port 7079 missing from feast-spark-driver service"
    return f"feast-spark-driver: ports {sorted(ports)}"

check(1, "Spark driver Service (k8s://)", check_spark_driver_svc)

# 1.8 materialize_incremental smoke test — spawns k8s:// executor, processes 1 feature view
def check_materialize():
    pods = v1.list_namespaced_pod(
        NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
        field_selector="status.phase=Running",
    ).items
    pod_name = pods[0].metadata.name
    script = f"""
import sys, time
sys.path.insert(0, "{FEAST_FEATURE_REPO_ON_POD}")
from datetime import datetime
from feast import FeatureStore

store = FeatureStore(repo_path="{FEAST_FEATURE_REPO_ON_POD}")
t0 = time.time()
store.materialize_incremental(
    end_date=datetime(2026, 12, 31),
    feature_views=["item_metadata"],
)
elapsed = time.time() - t0
print(f"MATERIALIZE_OK:{{elapsed:.0f}}")
"""
    print("    (spawning k8s:// RAPIDS executor — may take 1-2 min)...")
    resp = stream(
        v1.connect_get_namespaced_pod_exec,
        pod_name, NAMESPACE, container="offline",
        command=["python3", "-u", "-c", script],
        stderr=True, stdout=True, stdin=False,
        _request_timeout=600,
    )
    if "MATERIALIZE_OK:" in resp:
        elapsed = resp.split("MATERIALIZE_OK:")[1].split()[0]
        return f"item_metadata materialized in {elapsed}s (k8s:// executor worked)"
    for pattern in ["UNSUPPORTED_DATASOURCE", "ParseException", "ValueError", "connection refused"]:
        if pattern in resp:
            raise RuntimeError(f"Materialization failed: {pattern} — check spark.sql.runSQLOnFiles in offline_store config")
    raise RuntimeError(f"Unexpected output: {resp[-500:]}")

check(1, "materialize_incremental (k8s://)", check_materialize)

# 1.9 Verify online features written to Redis after materialize
def check_online_after_materialize():
    import redis as _redis
    r = _redis.Redis(host=REDIS_HOST, port=REDIS_PORT,
                     password=REDIS_PASSWORD or None, socket_timeout=5)
    from feast import FeatureStore
    store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)
    result = store.get_online_features(
        features=["item_metadata:item_category", "item_metadata:item_title"],
        entity_rows=[{"item_id": "B09V3KXJPB"}],
    ).to_dict()
    cat = result.get("item_category", [None])[0]
    title = result.get("item_title", [None])[0]
    if cat or title:
        return f"item_category={cat}, title={str(title)[:40]}..."
    return "⚠ item found but features are None (may need full materialize)"

check(1, "Online features in Redis", check_online_after_materialize)

# 1.10 get_historical_features with Python-mode BFV UDFs (patched SDK)
def check_historical_features():
    pods = v1.list_namespaced_pod(
        NAMESPACE, label_selector="feast.dev/name=smartshop-feast",
        field_selector="status.phase=Running",
    ).items
    pod_name = pods[0].metadata.name
    script = f"""
import sys
sys.path.insert(0, "{FEAST_FEATURE_REPO_ON_POD}")
from feast import FeatureStore
from pyspark.sql import SparkSession, Row
from datetime import datetime

store = FeatureStore(repo_path="{FEAST_FEATURE_REPO_ON_POD}")
from feast.infra.offline_stores.contrib.spark_offline_store.spark import (
    get_spark_session_or_start_new_with_repoconfig,
)
spark = get_spark_session_or_start_new_with_repoconfig(store.config.offline_store)
spark.conf.set("spark.sql.runSQLOnFiles", "true")

entity_df = spark.createDataFrame([
    Row(user_id="AEYORY24FHQJG", event_timestamp=datetime.now()),
], schema="user_id STRING, event_timestamp TIMESTAMP")

job = store.get_historical_features(
    entity_df=entity_df,
    features=["user_features:user_avg_rating", "user_features:user_review_count"],
)
result = job.to_spark_df()
cols = result.columns
n = result.count()
vals = result.first()
spark.stop()

if n > 0 and vals["user_avg_rating"] is not None:
    print(f"HISTORICAL_OK:cols={{cols}},avg_rating={{vals['user_avg_rating']:.2f}},count={{vals['user_review_count']}}")
else:
    print(f"HISTORICAL_FAIL:n={{n}},cols={{cols}}")
"""
    resp = stream(
        v1.connect_get_namespaced_pod_exec,
        pod_name, NAMESPACE, container="offline",
        command=["python3", "-u", "-c", script],
        stderr=True, stdout=True, stdin=False,
        _request_timeout=300,
    )
    if "HISTORICAL_OK:" in resp:
        detail = resp.split("HISTORICAL_OK:")[1].split("\n")[0]
        return f"UDF executed, {detail}"
    if "HISTORICAL_FAIL:" in resp:
        raise RuntimeError(f"get_historical_features returned nulls: {resp[-300:]}")
    raise RuntimeError(f"Unexpected: {resp[-500:]}")

check(1, "get_historical_features (patched BFV)", check_historical_features)

---
## Phase 2: Training

In [ ]:
import fsspec, io
import pandas as pd

print("Phase 2: Training\n")

PERSONAS_IDS = [
    ("Tech Enthusiast", "AGT45CD4STNWPKPJA57SBWNYC43A"),
    ("Book Lover", "AHF2B6SMLWPQ4RW7FFSHZ6YJBSCA"),
    ("Home Enthusiast", "AFCWX7L7FHBAFZBNKDBWOUHKPSVQ"),
]

# 2.1 Training data exists in S3
def check_training_data():
    fs, _ = fsspec.core.url_to_fs("s3://smartshop-features", endpoint_url=S3_ENDPOINT)
    files = [f for f in fs.ls("smartshop-features/interactions") if f.endswith(".parquet")]
    assert len(files) > 0, "No parquet files in s3://smartshop-features/interactions"
    total_size = sum(fs.info(f).get("size", 0) for f in files)
    return f"{len(files)} parquet files, {total_size / (1024**3):.1f} GB"

check(2, "Training data in S3", check_training_data)

# 2.2 Persona user IDs in training data
def check_persona_ids():
    fs, _ = fsspec.core.url_to_fs("s3://smartshop-features", endpoint_url=S3_ENDPOINT)
    files = sorted([f for f in fs.ls("smartshop-features/interactions") if f.endswith(".parquet")])
    target_ids = {uid for _, uid in PERSONAS_IDS}
    found = set()
    for f in files:
        with fs.open(f, "rb") as fh:
            chunk = pd.read_parquet(io.BytesIO(fh.read()), columns=["user_id"])
        found.update(target_ids & set(chunk["user_id"].unique()))
        if found == target_ids:
            break
    missing = target_ids - found
    assert not missing, f"Missing persona IDs: {missing}"
    return f"All 3 persona IDs found"

check(2, "Persona IDs in training data", check_persona_ids)

# 2.3 Kubeflow Trainer connectivity
def check_kubeflow():
    from kubeflow.trainer import TrainerClient
    from kubeflow.common.types import KubernetesBackendConfig
    trainer = TrainerClient(KubernetesBackendConfig(namespace=NAMESPACE))
    rt = trainer.get_runtime("torch-distributed")
    assert rt is not None
    return "torch-distributed runtime available"

check(2, "Kubeflow TrainerClient", check_kubeflow)

# 2.4 MLflow secret
def check_mlflow_secret():
    import base64
    secret = v1.read_namespaced_secret(MLFLOW_SECRET, NAMESPACE)
    uri = base64.b64decode(secret.data["MLFLOW_TRACKING_URI"]).decode()
    assert uri, "Empty MLFLOW_TRACKING_URI"
    return f"URI: {uri[:50]}..."

check(2, "MLflow secret accessible", check_mlflow_secret)

# 2.5 Model output bucket writable
def check_model_bucket():
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                      aws_access_key_id=AWS_KEY, aws_secret_access_key=AWS_SECRET,
                      region_name="us-east-1")
    s3.put_object(Bucket=S3_MODELS_BUCKET, Key="_validate_probe", Body=b"ok")
    s3.delete_object(Bucket=S3_MODELS_BUCKET, Key="_validate_probe")
    return f"s3://{S3_MODELS_BUCKET} writable"

check(2, "Model bucket writable", check_model_bucket)

---
## Phase 3: Serving

In [ ]:
import struct

print("Phase 3: Serving\n")

# 3.1 Model artifact exists
def check_model_artifact():
    fs, _ = fsspec.core.url_to_fs(f"s3://{S3_MODELS_BUCKET}", endpoint_url=S3_ENDPOINT)
    model_path = f"{S3_MODELS_BUCKET}/recommendation/best_model.pt"
    assert fs.exists(model_path), f"{model_path} not found"
    size_mb = fs.info(model_path).get("size", 0) / (1024 * 1024)
    return f"best_model.pt: {size_mb:.0f} MB"

check(3, "Model artifact in S3", check_model_artifact)

# 3.2 KServe client
def check_kserve():
    from kserve import KServeClient
    kserve = KServeClient()
    isvcs = kserve.get(namespace=NAMESPACE)
    names = [i["metadata"]["name"] for i in isvcs.get("items", [])]
    return f"{len(names)} InferenceServices: {', '.join(names) if names else 'none yet'}"

check(3, "KServe client", check_kserve)

# 3.3 Redis has materialized features
def check_redis_features():
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT,
                    password=REDIS_PASSWORD or None, decode_responses=False)
    results = {}
    for entity, fv_name in [("user_id", "user_features"), ("item_id", "item_features"), ("item_id", "item_metadata")]:
        cursor, keys = r.scan(0, match=f"*{entity}*smartshop".encode(), count=50)
        found = sum(1 for k in keys[:20] if any(fv_name.encode() in f for f in r.hgetall(k).keys()))
        results[fv_name] = found > 0
    populated = [k for k, v in results.items() if v]
    missing = [k for k, v in results.items() if not v]
    if missing:
        return f"Populated: {populated}, MISSING: {missing} (run 01_data_pipeline first)"
    return f"All 3 feature views populated"

check(3, "Redis feature views", check_redis_features)

# 3.4 Feast serving config valid
def check_feast_serving_config():
    import pathlib
    cfg_path = pathlib.Path("feature_repo/feature_store_rec.yaml")
    assert cfg_path.exists(), f"{cfg_path} not found"
    content = cfg_path.read_text()
    assert "online_store" in content
    assert "redis" in content
    assert "entity_key_serialization_version: 3" in content
    return "feature_store_rec.yaml valid"

check(3, "Feast serving config", check_feast_serving_config)

# 3.5 Feast online features smoke test
def check_feast_online():
    from feast import FeatureStore
    store = FeatureStore(fs_yaml_file=FEAST_CLIENT_CONFIG)
    result = store.get_online_features(
        features=["user_features:user_review_count", "user_features:user_primary_category"],
        entity_rows=[{"user_id": "AGT45CD4STNWPKPJA57SBWNYC43A"}],
    ).to_dict()
    count = result.get("user_review_count", [None])[0]
    cat = result.get("user_primary_category", [None])[0]
    if count is None:
        return "⚠ Tech persona not in Redis yet (run materialize first)"
    return f"Tech persona: {count} reviews, category={cat}"

check(3, "Feast online lookup (persona)", check_feast_online)

---
## Phase 4: Serving Endpoints (if deployed)

In [ ]:
import requests
import urllib3
urllib3.disable_warnings()

print("Phase 4: Live Endpoints\n")

REC_BASE = f"http://smartshop-rec-predictor.{NAMESPACE}.svc.cluster.local:8000"
LLM_BASE = f"http://smartshop-llm-predictor.{NAMESPACE}.svc.cluster.local:8000"
RAG_BASE = f"http://smartshop-rag-predictor.{NAMESPACE}.svc.cluster.local:8000"

# 4.1 Rec health
def check_rec_health():
    r = requests.get(f"{REC_BASE}/health", timeout=5)
    assert r.status_code == 200
    data = r.json()
    n_items = data.get("num_items")
    items_str = f"{n_items:,}" if n_items is not None else "N/A"
    return f"model_loaded={data.get('model_loaded')}, items={items_str}, feast={data.get('feast_connected')}"

check(4, "Rec service health", check_rec_health)

# 4.2 Rec predict
def check_rec_predict():
    r = requests.post(f"{REC_BASE}/v1/models/smartshop-rec:predict",
                      json={"user_id": "AGT45CD4STNWPKPJA57SBWNYC43A", "top_k": 3}, timeout=10)
    assert r.status_code == 200
    recs = r.json()["recommendations"]
    has_meta = any(rec.get("title") for rec in recs)
    return f"{len(recs)} recs, metadata={'YES' if has_meta else 'NO'}"

check(4, "Rec predict", check_rec_predict)

# 4.3 Rec user-profile
def check_rec_profile():
    r = requests.post(f"{REC_BASE}/v1/models/smartshop-rec:user-profile",
                      json={"user_id": "AGT45CD4STNWPKPJA57SBWNYC43A"}, timeout=5)
    assert r.status_code == 200
    profile = r.json().get("profile", {})
    return f"review_count={profile.get('review_count')}, category={profile.get('primary_category')}"

check(4, "Rec user-profile", check_rec_profile)

# 4.4 LLM health
def check_llm_health():
    r = requests.get(f"{LLM_BASE}/health", timeout=5)
    assert r.status_code == 200
    return "healthy"

check(4, "LLM service health", check_llm_health)

# 4.5 RAG health
def check_rag_health():
    r = requests.get(f"{RAG_BASE}/health", timeout=5)
    assert r.status_code == 200
    return "healthy"

check(4, "RAG service health", check_rag_health)

---
## Summary

In [ ]:
print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

passed = sum(1 for _, _, ok, _, _ in _results if ok)
failed = sum(1 for _, _, ok, _, _ in _results if not ok)
total_ms = sum(ms for _, _, _, _, ms in _results)

for phase in sorted(set(p for p, _, _, _, _ in _results)):
    phase_results = [(n, ok, msg) for p, n, ok, msg, _ in _results if p == phase]
    phase_pass = sum(1 for _, ok, _ in phase_results if ok)
    phase_total = len(phase_results)
    label = {1: "Data Pipeline", 2: "Training", 3: "Serving", 4: "Endpoints"}[phase]
    status = "✓" if phase_pass == phase_total else "⚠"
    print(f"\n  {status} Phase {phase}: {label} — {phase_pass}/{phase_total} passed")
    for name, ok, msg in phase_results:
        if not ok:
            print(f"      ✗ {name}: {msg[:100]}")

print(f"\n{'=' * 70}")
color = '\033[92m' if failed == 0 else '\033[93m'
print(f"{color}{passed}/{passed+failed} checks passed in {total_ms/1000:.1f}s\033[0m")

if failed > 0:
    print(f"\n⚠ {failed} check(s) failed — review above before running full pipeline.")
    print("  Phase 4 failures are expected if services aren't deployed yet.")
else:
    print("\n✓ All checks passed — pipeline is ready to run.")